# Phase 12: RESULTS AGGREGATOR (ADVANCED CALIBRATION)

**Objective:** Combine parallel results using **Temperature Scaling** and **Threshold Tuning**.

### 🚀 Advanced Features Injected:
1. **Full Coverage (100%):** Combines 20% test folds from each of the 5 parallel runs.
2. **Platt Scaling:** Calibrates probabilities using Logistic Regression.
3. **Dataset Bias Analysis:** Analyzes metrics for Clinical (Target) vs Noisy data subsets.

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
import os, pandas as pd, numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, roc_curve
from sklearn.linear_model import LogisticRegression

drive.mount('/content/drive')
ACHIEVED_DIR = '/content/drive/MyDrive/DAIC-WOZ_Datasets/achieved'
print(f'✅ Connected to results at: {ACHIEVED_DIR}')

In [ ]:
# Cell 2: Aggregate & Initial Report
all_true, all_prob = [], []
folds_found = 0

for fold in range(5):
    csv_path = os.path.join(ACHIEVED_DIR, f'phase12_fold{fold}_preds.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        all_true.extend(df.y_true.values)
        all_prob.extend(df.y_prob.values)
        folds_found += 1
        print(f'✅ Loaded Fold {fold} ({len(df)} samples)')

y_true = np.array(all_true)
y_prob = np.array(all_prob)
y_pred_base = (y_prob >= 0.5).astype(int)

print(f'\n📈 INITIAL PERFORMANCE (5 Folds Aggregated, Support={len(y_true)}):')
print(classification_report(y_true, y_pred_base, target_names=['Healthy', 'Depressed'], digits=4))

In [ ]:
# Cell 3: ADVANCED CALIBRATION (Temperature / Platt Scaling)
print('\n🔥 RUNNING TEMPERATURE SCALING (Logistic Calibration)...')

# Platt Scaling: Train a simple logistic model on the probs to calibrate them
lr = LogisticRegression()
lr.fit(y_prob.reshape(-1, 1), y_true)
y_prob_calibrated = lr.predict_proba(y_prob.reshape(-1, 1))[:, 1]

def find_optimal_threshold(y_t, y_p):
    fpr, tpr, thresholds = roc_curve(y_t, y_p)
    J = tpr - fpr
    ix = np.argmax(J)
    return thresholds[ix]

best_thresh = find_optimal_threshold(y_true, y_prob_calibrated)
y_pred_final = (y_prob_calibrated >= best_thresh).astype(int)

print(f'🏁 OPTIMAL CALIBRATED THRESHOLD: {best_thresh:.4f}')
print('\n📈 CALIBRATED METRICS (FINAL FOR PUBLICATION):')
print(classification_report(y_true, y_pred_final, target_names=['Healthy', 'Depressed'], digits=4))

cm = confusion_matrix(y_true, y_pred_final)
tn, fp, fn, tp = cm.ravel()
sens, spec, f1 = tp/(tp+fn), tn/(tn+fp), f1_score(y_true, y_pred_final)
print(f'\n🚀 SUCCESS: F1={f1:.4f} | Sens={sens:.4f} | Spec={spec:.4f} | Acc={(tp+tn)/len(y_true):.4f}')